In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ndavuti/picmus-simulation-resolution-distortion-uff/PICMUS_simulation_resolution_distortion.uff
/kaggle/input/datasets/ndavuti/alpinion-l3-8-cpwc-hypoechoic-uff/Alpinion_L3-8_CPWC_hypoechoic.uff
/kaggle/input/datasets/ndavuti/l7-cpwc-thegb-uff/L7_CPWC_TheGB.uff
/kaggle/input/datasets/ndavuti/picmus-carotid-cross-uff/PICMUS_carotid_cross.uff


In [2]:
# ==================== CELL 2 ====================
!pip install git+https://github.com/magnusdk/pyuff_ustb.git
!pip install opencv-python networkx

  Cloning https://github.com/magnusdk/pyuff_ustb.git to /tmp/pip-req-build-gjon5ovb
  Running command git clone --filter=blob:none --quiet https://github.com/magnusdk/pyuff_ustb.git /tmp/pip-req-build-gjon5ovb
  Resolved https://github.com/magnusdk/pyuff_ustb.git to commit d246f0841417b10c6719f23926c0f6990487ac30
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyuff_ustb: filename=pyuff_ustb-3.0.0-py3-none-any.whl size=37822 sha256=e161da6941b9b9e30e93ca33a24b5a213280e7c672b1e7d177ccdb6a16e5457c
  Stored in directory: /tmp/pip-ephem-wheel-cache-ahliw9na/wheels/7f/d7/0e/cdb155d6702a67dd53fa556088ab13fc3527230f81e951cc20
Successfully built pyuff_ustb


In [3]:
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, find_peaks
from scipy.ndimage import median_filter
from scipy.interpolate import UnivariateSpline
import networkx as nx
import ipywidgets as widgets
from IPython.display import display, clear_output

# =============================================================================
# 1. Find all .uff files in Kaggle directories
# =============================================================================
def find_all_uff_files():
    """Return a list of (display_name, full_path) for every .uff file."""
    files = []
    search_roots = ["/kaggle/input", "/kaggle/working"]
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                if f.endswith(".uff"):
                    full_path = os.path.join(dirpath, f)
                    # Use the file name as the display label
                    files.append((f, full_path))
    return files

all_files = find_all_uff_files()
if not all_files:
    raise FileNotFoundError("No .uff files found. Please add a dataset to this notebook.")

# Create dropdown widget
file_options = [(name, path) for name, path in all_files]
dropdown = widgets.Dropdown(
    options=file_options,
    description='Dataset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

# Button and output area
run_button = widgets.Button(description="Run Analysis", button_style='success')
output_area = widgets.Output()

# Display widgets
display(dropdown, run_button, output_area)

# =============================================================================
# 2. The core processing function (called when button clicked)
# =============================================================================
def process_dataset(uff_path):
    """Run the full Vortex Engine pipeline on a given .uff file."""
    with output_area:
        clear_output(wait=True)
        print(f"Processing: {uff_path}\n")

        # ---- Load data ----
        with h5py.File(uff_path, 'r') as f:
            data_full = f['channel_data']['data'][:]
            if data_full.dtype.names:
                data_full = data_full['r'] + 1j * data_full['i']
            geom = f['channel_data']['probe']['geometry'][:]
            x_pos = geom[0, :].ravel()
            c0 = float(np.asarray(f['channel_data']['sequence']['sequence_0001']['sound_speed'][()]).ravel()[0])
            fs = float(np.asarray(f['channel_data']['sampling_frequency'][()]).ravel()[0])
            t0 = float(np.asarray(f['channel_data']['initial_time'][()]).ravel()[0])

        # Extract first transmit
        if data_full.ndim == 4:
            raw = data_full[:, :, 0, 0]
        elif data_full.ndim == 3:
            if data_full.shape[2] > data_full.shape[0]:
                raw = data_full[0, :, :].T
            else:
                raw = data_full[:, :, 0]
        else:
            raw = data_full[:, :]
        raw = np.array(raw)
        N_samples, N_ch = raw.shape
        depth_axis = ((np.arange(N_samples) / fs) * c0 / 2 + t0 * c0 / 2).ravel()

        print(f"Samples: {N_samples}, Channels: {N_ch}")

        # ---- Beamforming ----
        iq_orig = raw
        x_img = np.linspace(x_pos.min(), x_pos.max(), 256)
        z_img = depth_axis
        X, Z = np.meshgrid(x_img, z_img)
        b_mode = np.zeros((len(z_img), len(x_img)), dtype=np.complex64)
        for ch in range(N_ch):
            dist = np.sqrt((X - x_pos[ch])**2 + Z**2)
            delays = 2 * dist / c0 * fs
            idx_floor = np.floor(delays).astype(int)
            idx_ceil = idx_floor + 1
            valid = (idx_floor >= 0) & (idx_ceil < N_samples)
            idx_floor = np.clip(idx_floor, 0, N_samples-1)
            idx_ceil = np.clip(idx_ceil, 0, N_samples-1)
            w = delays - idx_floor
            contrib = (1 - w) * iq_orig[idx_floor, ch] + w * iq_orig[idx_ceil, ch]
            contrib[~valid] = 0
            b_mode += contrib
        b_mode_env = np.abs(b_mode)
        b_mode_norm = b_mode_env / np.max(b_mode_env)
        b_dB = 20 * np.log10(b_mode_norm + 1e-3)

        # ---- Green Mask ----
        t = np.arange(N_samples) / fs
        gain = np.exp(+t * 2000); gain = gain / np.max(gain); gain = gain.reshape(-1, 1)
        iq_tgc = iq_orig * gain.astype(np.complex64)
        cut_samples = int((2 * 10e-3 / c0) * fs)
        iq_cut = iq_tgc.copy(); iq_cut[:cut_samples, :] = 0
        amp = np.abs(iq_cut).astype(np.float32)
        p_val = np.percentile(amp.flatten(), 85)
        green_mask = amp > p_val

        # ---- Peak detection & graph ----
        LAT_MIN, LAT_MAX = -10e-3, 10e-3
        DEPTH_BROAD_MIN = 10e-3
        DEPTH_BROAD_MAX = depth_axis[-1] * 0.9
        idx_broad_min = np.searchsorted(depth_axis, DEPTH_BROAD_MIN)
        idx_broad_max = np.searchsorted(depth_axis, DEPTH_BROAD_MAX)

        all_peaks_x, all_peaks_z, all_peaks_ch = [], [], []
        for ch in range(N_ch):
            if x_pos[ch] < LAT_MIN or x_pos[ch] > LAT_MAX: continue
            aline = amp[idx_broad_min:idx_broad_max, ch]
            aline_masked = aline * green_mask[idx_broad_min:idx_broad_max, ch].astype(np.float32)
            if np.max(aline_masked) == 0: continue
            peaks, _ = find_peaks(aline_masked, prominence=0.1*np.max(aline_masked), distance=10)
            for p in peaks:
                abs_idx = p + idx_broad_min
                all_peaks_x.append(x_pos[ch])
                all_peaks_z.append(depth_axis[abs_idx])
                all_peaks_ch.append(ch)
        all_peaks_x = np.array(all_peaks_x); all_peaks_z = np.array(all_peaks_z); all_peaks_ch = np.array(all_peaks_ch)

        # Adaptive threshold
        shallowest = np.full(N_ch, np.nan)
        for ch in range(N_ch):
            if x_pos[ch] < LAT_MIN or x_pos[ch] > LAT_MAX: continue
            green_idx = np.where(green_mask[:, ch])[0]
            valid = green_idx[green_idx >= idx_broad_min]
            if len(valid) > 0: shallowest[ch] = depth_axis[valid[0]]
        valid_shallow = shallowest[np.isfinite(shallowest)]
        std_shallow = np.std(valid_shallow) if len(valid_shallow) > 5 else 0.5e-3
        DEPTH_THRESH = np.clip(3.0 * std_shallow, 0.5e-3, 1.5e-3)

        G = nx.Graph()
        for i, (x, z, ch) in enumerate(zip(all_peaks_x, all_peaks_z, all_peaks_ch)):
            G.add_node(i, x=x, z=z, ch=ch)
        ch_dict = {}
        for i, ch in enumerate(all_peaks_ch): ch_dict.setdefault(ch, []).append(i)
        for ch in ch_dict:
            if ch + 1 not in ch_dict: continue
            for i in ch_dict[ch]:
                zi = all_peaks_z[i]
                for j in ch_dict[ch+1]:
                    if abs(zi - all_peaks_z[j]) < DEPTH_THRESH:
                        G.add_edge(i, j)
        components = list(nx.connected_components(G))

        # ---- Auto‑click ----
        TOLERANCE = 4.0e-3
        CONTRAST_NEAR, CONTRAST_FAR = 2e-3, 6e-3
        def contrast_of_component(comp):
            comp_x = all_peaks_x[list(comp)]; comp_z = all_peaks_z[list(comp)]
            contrasts = []
            for x, z in zip(comp_x, comp_z):
                ch = np.argmin(np.abs(x_pos - x))
                idx_wall = np.searchsorted(depth_axis, z)
                wall_bright = b_mode_norm[idx_wall, ch]
                lumen_min_idx = np.searchsorted(depth_axis, z + CONTRAST_NEAR)
                lumen_max_idx = np.searchsorted(depth_axis, z + CONTRAST_FAR)
                if lumen_max_idx > lumen_min_idx:
                    lumen_bright = np.mean(b_mode_norm[lumen_min_idx:lumen_max_idx, ch])
                else: lumen_bright = 0.0
                contrasts.append(wall_bright - lumen_bright)
            return np.mean(contrasts)

        click_x = 0.0
        max_depth = depth_axis[-1] * 0.9
        candidate_depths = np.linspace(max_depth * 0.2, max_depth * 0.8, 7)

        candidate_ridges = []
        for click_z in candidate_depths:
            for comp in components:
                if len(comp) < 5: continue
                comp_x = all_peaks_x[list(comp)]; comp_z = all_peaks_z[list(comp)]
                dists = np.sqrt((comp_x - click_x)**2 + (comp_z - click_z)**2)
                min_dist = np.min(dists)
                if min_dist > TOLERANCE: continue
                local_mask = dists < TOLERANCE
                local_x = comp_x[local_mask]; local_z = comp_z[local_mask]
                if len(local_x) < 5: continue
                contrast = contrast_of_component(comp)
                median_depth = np.median(local_z)
                candidate_ridges.append((click_z, local_x, local_z, contrast, median_depth))

        if not candidate_ridges:
            print("⚠️ No ridges found. Falling back to largest component.")
            largest = max(components, key=len)
            best_ridge = (all_peaks_x[list(largest)], all_peaks_z[list(largest)])
            best_click = np.median(best_ridge[1])
        else:
            valid_ridges = [r for r in candidate_ridges if r[3] > 0.01]
            if not valid_ridges: valid_ridges = candidate_ridges
            best = max(valid_ridges, key=lambda r: r[4])
            best_click = best[0]
            best_ridge = (best[1], best[2])

        ridge_x, ridge_z = best_ridge

        # ---- Envelope & spline ----
        sort_idx = np.argsort(ridge_x)
        ridge_x = ridge_x[sort_idx]; ridge_z = ridge_z[sort_idx]
        unique_x, unique_idx = np.unique(ridge_x, return_index=True)
        unique_z = ridge_z[unique_idx]
        unique_z = median_filter(unique_z, size=3)
        if len(unique_x) > 2:
            spline = UnivariateSpline(unique_x, unique_z, s=1e-5)
            x_fit = np.linspace(unique_x.min(), unique_x.max(), 200)
            z_fit = spline(x_fit)
        else: x_fit, z_fit = unique_x, unique_z

        # ---- B‑mode peaks ----
        idx_min_bm = np.searchsorted(depth_axis, DEPTH_BROAD_MIN)
        idx_max_bm = np.searchsorted(depth_axis, DEPTH_BROAD_MAX)
        bm_peak_idx = np.argmax(b_mode_env[idx_min_bm:idx_max_bm, :], axis=0) + idx_min_bm
        bm_peak_z = z_img[bm_peak_idx]; bm_peak_x = x_img
        bm_mask = (x_img >= LAT_MIN) & (x_img <= LAT_MAX)
        bm_x_gated = x_img[bm_mask]; bm_z_gated = bm_peak_z[bm_mask]

        # ---- Metrics ----
        def metrics(px, pz, lat_range):
            if len(px) == 0: return 0,0,0
            lat_cov = (px.max()-px.min())/(lat_range[1]-lat_range[0])
            depth_rng = pz.max()-pz.min()
            depth_std = np.std(pz)
            return lat_cov, depth_rng, depth_std
        v_cov, v_rng, v_std = metrics(ridge_x, ridge_z, (LAT_MIN, LAT_MAX))
        b_cov, b_rng, b_std = metrics(bm_x_gated, bm_z_gated, (LAT_MIN, LAT_MAX))

        print("\n" + "="*60)
        print("QUANTITATIVE COMPARISON")
        print("="*60)
        print(f"{'Metric':<30} {'Vortex':<15} {'B‑mode Peaks':<15}")
        print("-"*60)
        print(f"{'Points':<30} {len(ridge_x):<15} {len(bm_x_gated):<15}")
        print(f"{'Lateral coverage':<30} {v_cov:<15.2f} {b_cov:<15.2f}")
        print(f"{'Depth range (mm)':<30} {v_rng*1e3:<15.2f} {b_rng*1e3:<15.2f}")
        print(f"{'Depth std (mm)':<30} {v_std*1e3:<15.2f} {b_std*1e3:<15.2f}")
        print("="*60)

        # ---- Figure ----
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16,6))
        ax1.imshow(b_dB, extent=[x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3],
                   cmap='gray', vmin=-40, vmax=0, aspect='auto')
        ax1.scatter(bm_x_gated*1e3, bm_z_gated*1e3, c='orange', s=8, alpha=0.6, label='B‑mode peaks')
        ax1.set_title('Standard B‑mode Peak Detection'); ax1.set_xlabel('Lateral [mm]'); ax1.set_ylabel('Depth [mm]')
        ax1.legend(loc='upper right')

        ax2.imshow(b_dB, extent=[x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3],
                   cmap='gray', vmin=-40, vmax=0, aspect='auto')
        ax2.scatter(ridge_x*1e3, ridge_z*1e3, c='lime', s=15, alpha=0.7, label='Vortex ridge')
        if len(x_fit) > 0: ax2.plot(x_fit*1e3, z_fit*1e3, 'lime', lw=2, label='Fitted wall')
        ax2.scatter(click_x*1e3, best_click*1e3, c='red', marker='x', s=100, label='Auto‑click')
        ax2.set_title('Vortex Engine (Fully Automatic)'); ax2.set_xlabel('Lateral [mm]'); ax2.set_ylabel('')
        ax2.legend(loc='upper right')

        plt.suptitle(f'Dataset: {os.path.basename(uff_path)}', fontsize=14)
        plt.tight_layout()
        plt.show()

# =============================================================================
# 3. Button click handler
# =============================================================================
def on_button_clicked(b):
    selected_path = dropdown.value
    process_dataset(selected_path)

run_button.on_click(on_button_clicked)

Dropdown(description='Dataset:', layout=Layout(width='70%'), options=(('PICMUS_simulation_resolution_distortio…

Button(button_style='success', description='Run Analysis', style=ButtonStyle())

Output()